# Label Studio on SPCS — Hands-On Lab

Deploy Label Studio, an open-source data labeling platform, on Snowpark Container Services with Snowflake Postgres as the backend database.

**Duration:** 30-45 minutes

**What you'll learn:**
- Deploy a containerized web application on SPCS
- Use Snowflake Postgres as an application database backend
- Mount Snowflake stages as data volumes
- Access SPCS services via public endpoints
- Export annotations back to Snowflake tables

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    Snowflake Account                         │
│                                                              │
│  ┌──────────────────┐     ┌─────────────────────────────┐  │
│  │  Snowflake        │     │  SPCS Compute Pool           │  │
│  │  Postgres         │◄────│  ┌─────────────────────┐    │  │
│  │  (Label Studio DB)│     │  │  Label Studio        │    │  │
│  └──────────────────┘     │  │  Container            │    │  │
│                            │  └──────────┬────────────┘    │  │
│  ┌──────────────────┐     │             │                  │  │
│  │  @DATA_STAGE      │◄───┼─────────────┘                  │  │
│  │  (source files)   │     │                               │  │
│  └──────────────────┘     └─────────────────────────────┘  │
│                                                              │
│  ┌──────────────────┐                                       │
│  │  LABELED_DATA     │  ← Exported annotations              │
│  │  (table)          │                                      │
│  └──────────────────┘                                       │
└─────────────────────────────────────────────────────────────┘
```

## Prerequisites

Before starting this lab, ensure:
1. You have run `setup.sql` to create all required objects
2. The compute pool is in ACTIVE or IDLE state
3. The Postgres instance is in READY state
4. You have saved the Postgres credentials from the CREATE POSTGRES INSTANCE output

In [ ]:
-- Verify compute pool status
DESCRIBE COMPUTE POOL LABEL_STUDIO_SPCS.APP.LABEL_STUDIO_POOL;

In [ ]:
-- Verify Postgres instance status
DESCRIBE POSTGRES INSTANCE LABEL_STUDIO_PG;

## Step 1: Build and Push the Container Image

Run these commands in your local terminal (not in this notebook):

```bash
# Authenticate Docker with Snowflake image registry
snow spcs image-registry login

# Get your registry URL
snow spcs image-registry url

# Build the image (from the lab/ directory)
docker build -t <registry_url>/label_studio_spcs/app/label_studio_repo/label-studio:latest .

# Push to Snowflake
docker push <registry_url>/label_studio_spcs/app/label_studio_repo/label-studio:latest
```

> **Note:** Replace `<registry_url>` with the output of `snow spcs image-registry url`

## Step 2: Update Service Spec with Postgres Credentials

Before deploying, update `label-studio-spec.yaml` with the Postgres host and password from the CREATE POSTGRES INSTANCE output, then upload it:

```bash
# Upload the spec to the stage
snow stage copy label-studio-spec.yaml @LABEL_STUDIO_SPCS.APP.SPEC_STAGE/
```

In [ ]:
-- Create the Label Studio service
USE DATABASE LABEL_STUDIO_SPCS;
USE SCHEMA APP;

CREATE SERVICE IF NOT EXISTS LABEL_STUDIO_SERVICE
  IN COMPUTE POOL LABEL_STUDIO_POOL
  FROM @SPEC_STAGE
  SPECIFICATION_FILE = 'label-studio-spec.yaml'
  EXTERNAL_ACCESS_INTEGRATIONS = (LABEL_STUDIO_EAI)
  MIN_INSTANCES = 1
  MAX_INSTANCES = 1;

In [ ]:
-- Check service status (wait until status is READY)
SELECT SYSTEM$GET_SERVICE_STATUS('LABEL_STUDIO_SERVICE');

In [ ]:
-- View container logs for debugging
CALL SYSTEM$GET_SERVICE_LOGS('LABEL_STUDIO_SERVICE', 0, 'label-studio', 50);

In [ ]:
-- Get the public endpoint URL
SHOW ENDPOINTS IN SERVICE LABEL_STUDIO_SERVICE;

## Step 3: Access the Label Studio UI

1. Copy the endpoint URL from the query above (the `ingress_url` column)
2. Open it in your browser
3. You'll authenticate via Snowflake OAuth
4. Create an account in Label Studio (first user becomes admin):
   - Email: your email
   - Password: choose a password

> **Note:** The first startup may take 1-2 minutes while Label Studio initializes its database schema in Postgres.

## Step 4: Create a Labeling Project

In the Label Studio UI:
1. Click **Create Project**
2. Name it "Sentiment Analysis"
3. Under **Labeling Setup**, select **Natural Language Processing → Sentiment Analysis**
4. This gives you Positive/Negative/Neutral labels

The project is now ready for data import.

## Step 5: Import Data from Snowflake Stage

The sample data was uploaded to `@DATA_STAGE` during setup. In Label Studio:
1. Go to your project → **Settings** → **Cloud Storage**
2. Select **Add Source Storage** → **Local Files**
3. Set the path to `/label-studio/files`
4. Click **Sync Storage**

Alternatively, you can export tasks from Snowflake for direct import:

In [ ]:
-- Export labeling tasks as JSON for Label Studio import
SELECT ARRAY_AGG(
    OBJECT_CONSTRUCT(
        'id', TASK_ID,
        'data', OBJECT_CONSTRUCT('text', TEXT_CONTENT)
    )
) AS tasks
FROM LABEL_STUDIO_SPCS.APP.LABELING_TASKS;

## Step 6: Annotate Data

1. Open your project in the Label Studio UI
2. Click on a task to open the annotation interface
3. Read the text and select the appropriate sentiment label
4. Click **Submit** to save the annotation
5. Repeat for several tasks

> **Tip:** Use keyboard shortcuts (1, 2, 3) to speed up labeling

## Step 7: Export Annotations Back to Snowflake

Rather than manually exporting from the UI, we can query Label Studio's Postgres database directly from Snowflake using a Python stored procedure with `psycopg2`.

This approach:
- Requires no intermediate files or stages
- Works with the BURST_S Postgres instance (no pg_lake needed)
- Can be automated via Snowflake Tasks for scheduled syncs
- Gives full control over which tables and fields to extract

The procedure connects to Postgres, queries Label Studio's internal `task`, `task_completion`, and `auth_user` tables, and returns the results as a Snowflake table.

In [ ]:
-- Create a stored procedure that queries Label Studio's Postgres backend directly
CREATE OR REPLACE PROCEDURE LABEL_STUDIO_SPCS.APP.EXPORT_ANNOTATIONS(PROJECT_ID INT)
RETURNS TABLE (TASK_ID INT, TEXT_CONTENT VARCHAR, LABEL VARCHAR, ANNOTATOR VARCHAR, ANNOTATED_AT TIMESTAMP_NTZ)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.12'
PACKAGES = ('snowflake-snowpark-python', 'psycopg2-binary')
EXTERNAL_ACCESS_INTEGRATIONS = (PG_ACCESS_EAI)
SECRETS = ('pg_cred' = PG_CREDENTIALS)
HANDLER = 'run'
AS $$
import psycopg2
import _snowflake
import json
from datetime import datetime

def run(session, project_id):
    creds = json.loads(_snowflake.get_generic_secret_string('pg_cred'))
    conn = psycopg2.connect(
        host=creds['host'],
        port=int(creds['port']),
        dbname=creds['dbname'],
        user=creds['user'],
        password=creds['password'],
        sslmode='require'
    )
    cur = conn.cursor()
    cur.execute("""
        SELECT t.id,
               t.data->>'text' AS text_content,
               tc.result->0->'value'->>'choices' AS label,
               u.email AS annotator,
               tc.created_at
        FROM task t
        JOIN task_completion tc ON tc.task_id = t.id
        JOIN auth_user u ON u.id = tc.completed_by_id
        WHERE t.project_id = %s
    """, (project_id,))
    rows = cur.fetchall()
    conn.close()
    return session.create_dataframe(rows,
        schema=['TASK_ID', 'TEXT_CONTENT', 'LABEL', 'ANNOTATOR', 'ANNOTATED_AT'])
$$;

## Step 8: Sync Annotations to Snowflake

Call the procedure with your project ID (typically 1 for the first project) and insert the results into the `LABELED_DATA` table:

In [ ]:
-- Export annotations from Label Studio Postgres into Snowflake
-- Replace 1 with your actual project ID if different
INSERT INTO LABEL_STUDIO_SPCS.APP.LABELED_DATA
SELECT * FROM TABLE(LABEL_STUDIO_SPCS.APP.EXPORT_ANNOTATIONS(1));

-- Verify the export
SELECT * FROM LABEL_STUDIO_SPCS.APP.LABELED_DATA LIMIT 10;

## Step 9: Compare Human Labels with Cortex AI

Use the labeled data as ground truth to evaluate how well Cortex AI's classification matches human annotators:

In [ ]:
-- Compare human labels with Cortex AI classification
SELECT 
    l.TEXT_CONTENT,
    l.LABEL AS human_label,
    AI_CLASSIFY(l.TEXT_CONTENT, ['Positive', 'Negative', 'Neutral']):labels[0]::VARCHAR AS ai_label,
    CASE WHEN l.LABEL = AI_CLASSIFY(l.TEXT_CONTENT, ['Positive', 'Negative', 'Neutral']):labels[0]::VARCHAR 
         THEN 'Match' ELSE 'Mismatch' END AS agreement
FROM LABEL_STUDIO_SPCS.APP.LABELED_DATA l
WHERE l.LABEL IS NOT NULL;

## Cleanup

When you're done with the lab, clean up resources to avoid unnecessary costs:

In [ ]:
-- Drop the service
DROP SERVICE IF EXISTS LABEL_STUDIO_SPCS.APP.LABEL_STUDIO_SERVICE;

-- Suspend the compute pool
ALTER COMPUTE POOL LABEL_STUDIO_SPCS.APP.LABEL_STUDIO_POOL SUSPEND;

-- Optionally drop the Postgres instance (this deletes all data!)
-- DROP POSTGRES INSTANCE IF EXISTS LABEL_STUDIO_PG;

-- Optionally drop everything
-- DROP DATABASE IF EXISTS LABEL_STUDIO_SPCS;
-- DROP COMPUTE POOL IF EXISTS LABEL_STUDIO_POOL;

## Summary

In this lab you:
- Deployed Label Studio as a containerized service on SPCS
- Used Snowflake Postgres as a managed database backend
- Connected labeling data via Snowflake stage volumes
- Accessed the UI through a secure public endpoint
- Labeled data and compared with Cortex AI functions

**Key takeaways:**
- SPCS runs any containerized application — not just ML inference
- Snowflake Postgres eliminates the need for external database management
- Data never leaves the Snowflake security perimeter
- Labels produced can feed directly into Cortex AI evaluation pipelines